# DiT Fig.2 Generalization Sweep: Results Check

Clean readout for the Transformer/DiT version of the CAMELS HI memorization-to-generalization sweep.

This notebook assumes the Great Lakes jobs have already produced train, sample, PCA, and SSCD outputs under `results/nf_generalize_fig2_dit/`. It is meant to answer four practical questions:

1. Did every DiT training/sampling/evaluation output land where expected?
2. How did the DiT training loss evolve across data sizes?
3. Do DiT samples look reasonable, and do their one-point PDF / power spectra match real CAMELS slices?
4. Where does DiT move from training-slice copying to novel generation under PCA and SSCD nearest-neighbor tests, compared with the existing UNet sweep?


## tl;dr

Run all cells on Great Lakes after the DiT sweep completes. The notebook will print:

- a file audit for the 10 DiT dataset sizes, `2^6` through `2^15`;
- training-loss curves from checkpoint metrics;
- generated image grids across training-set size;
- one-point PDF and mean `P(k)` comparisons against real CAMELS slices;
- PCA and SSCD generalization curves;
- an estimated `N50`, the training-set size where the generalization score crosses 0.5;
- an optional DiT-vs-UNet comparison if the original Fig.2 tables are present;
- a capacity check asking whether DiT-base behaves like the parameter-matched UNet-128.

The `N50` number is a diagnostic, not a law: with one DiT architecture, it tells us where this model transitions, but not yet how Transformer capacity scales.


## Setup


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display


def find_project_dir() -> Path:
    env = os.environ.get('DIFFUSION_PROJECT_DIR')
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'scripts').exists() and (candidate / 'notebooks').exists():
            return candidate
    return here


PROJECT_DIR = find_project_dir()
SWEEP_NAME = 'nf_generalize_fig2_dit'
RESULTS_DIR = PROJECT_DIR / 'results' / SWEEP_NAME
TABLE_DIR = RESULTS_DIR / 'tables'
QUICKCHECK_DIR = RESULTS_DIR / 'quickcheck'
SAMPLE_DIR = RESULTS_DIR / 'samples'
MANIFEST_PATH = PROJECT_DIR / 'local' / SWEEP_NAME / 'manifest.json'
SAMPLE_LABEL = os.environ.get('SAMPLE_LABEL', 'dpm50')
SEED = int(os.environ.get('SEED', '123'))

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 300,
    'font.size': 14,
    'axes.labelsize': 15,
    'axes.titlesize': 16,
    'legend.fontsize': 12,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
})

print('PROJECT_DIR =', PROJECT_DIR)
print('RESULTS_DIR =', RESULTS_DIR)
print('SAMPLE_LABEL =', SAMPLE_LABEL)
print('SEED =', SEED)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

try:
    from simdiff_eval.io import as_nchw, load_real_from_config
    from simdiff_eval.metrics import batch_power_spectra, field_histogram
    SIMDIFF_EVAL_AVAILABLE = True
except Exception as exc:
    SIMDIFF_EVAL_AVAILABLE = False
    SIMDIFF_EVAL_ERROR = repr(exc)
    print('simdiff_eval unavailable; image/P(k) diagnostics will be skipped:', SIMDIFF_EVAL_ERROR)

PK_NBINS = int(os.environ.get('DIT_PK_NBINS', '30'))
MAX_GENERATED = int(os.environ.get('DIT_MAX_GENERATED', '512'))
MAX_REAL_RAW_CUBES = int(os.environ.get('DIT_MAX_REAL_RAW_CUBES', '16'))
DETAIL_TAGS = [x.strip() for x in os.environ.get('DIT_DETAIL_TAGS', 'd2p06,d2p08,d2p10,d2p12,d2p15').split(',') if x.strip()]
IMAGE_TAGS = [x.strip() for x in os.environ.get('DIT_IMAGE_TAGS', 'd2p06,d2p08,d2p10,d2p12,d2p15').split(',') if x.strip()]


## Context & Methods

- Model: DiT-base Transformer diffusion model on `128 x 128` CAMELS HI slices.
- Conditioning: this is the unconditional Fig.2-style sweep. The DiT implementation uses a single null class label internally because the diffusers DiT block requires `class_labels` for its adaLN path.
- Sweep: one run for each training-set size, `N_2D = 2^6, ..., 2^15`.
- Sampling: DPM-Solver, 50 steps, `512` generated slices per run.
- Diagnostics: nearest-neighbor similarity to real training slices in PCA and SSCD embedding spaces. High similarity means the generated sample is training-set-like; the plotted generalization score is high when generated fields are not unusually close to training slices.

Use this as a model-family comparison to the UNet Fig.2 sweep. It does not replace the full physical-fidelity checks.


## Data Audit


In [ ]:
def read_json(path: Path) -> Any | None:
    if not path.exists():
        return None
    with path.open() as f:
        return json.load(f)


def rel(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_DIR))
    except ValueError:
        return str(path)


def dataset_tag_from_name(name: str) -> str | None:
    m = re.search(r'd2p(\d+)', str(name))
    if not m:
        return None
    return 'd2p' + m.group(1)


def dataset_size_from_tag(tag: str | None) -> int | None:
    if not tag:
        return None
    m = re.match(r'd2p(\d+)', tag)
    if not m:
        return None
    return 2 ** int(m.group(1))


def sample_path_for(row: pd.Series) -> Path:
    raw = str(row.get('sample_path', '') or '')
    if raw:
        raw = raw.format(seed=SEED, sample_label=SAMPLE_LABEL)
        path = Path(raw)
        return path if path.is_absolute() else PROJECT_DIR / path
    run_name = row.get('run_name') or row.get('name')
    tag = row.get('dataset_tag') or dataset_tag_from_name(str(run_name))
    if tag is None:
        return SAMPLE_DIR / f'unknown_seed{SEED}_{SAMPLE_LABEL}.npz'
    return SAMPLE_DIR / f'nf_fig2_dit_base_{tag}_noaug_200k_seed{SEED}_{SAMPLE_LABEL}.npz'


manifest_obj = read_json(MANIFEST_PATH)
if manifest_obj is None:
    display(Markdown(f'**Missing manifest:** `{rel(MANIFEST_PATH)}`'))
    manifest_df = pd.DataFrame()
else:
    if isinstance(manifest_obj, dict):
        rows = manifest_obj.get('runs', [])
    elif isinstance(manifest_obj, list):
        rows = manifest_obj
    else:
        rows = []
    manifest_df = pd.DataFrame(rows)
    if 'run_name' not in manifest_df.columns and 'name' in manifest_df.columns:
        manifest_df['run_name'] = manifest_df['name']
    if 'dataset_tag' not in manifest_df.columns:
        manifest_df['dataset_tag'] = manifest_df['run_name'].map(dataset_tag_from_name)
    if 'dataset_size' not in manifest_df.columns:
        manifest_df['dataset_size'] = manifest_df['dataset_tag'].map(dataset_size_from_tag)
    manifest_df['sample_path_resolved'] = manifest_df.apply(sample_path_for, axis=1)
    manifest_df['sample_exists'] = manifest_df['sample_path_resolved'].map(Path.exists)
    manifest_df['sample_size_mb'] = manifest_df['sample_path_resolved'].map(lambda p: p.stat().st_size / 1024**2 if p.exists() else np.nan)

    show_cols = [c for c in [
        'run_name', 'dataset_tag', 'dataset_size', 'sample_exists', 'sample_size_mb',
        'config_path', 'output_dir', 'sample_path_resolved'
    ] if c in manifest_df.columns]
    display(manifest_df[show_cols].sort_values('dataset_size'))
    print(f"sample files present: {manifest_df['sample_exists'].sum()} / {len(manifest_df)}")

expected_tables = [
    TABLE_DIR / 'nf_generalize_fig2_dit_pca_full_nn_metrics.csv',
    TABLE_DIR / 'nf_generalize_fig2_dit_pca_full_nn_mode_norms.csv',
    TABLE_DIR / 'nf_generalize_fig2_dit_pca_full_nn_similarity_histograms.csv',
    TABLE_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_metrics.csv',
]
expected_figures = [
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_paper_style_gl_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_paper_style_gl_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_similarity_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_similarity_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_copy_fraction_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_copy_fraction_curves.png',
]

audit = pd.DataFrame({
    'path': [rel(p) for p in expected_tables + expected_figures],
    'kind': ['table'] * len(expected_tables) + ['figure'] * len(expected_figures),
    'exists': [p.exists() for p in expected_tables + expected_figures],
    'size_mb': [p.stat().st_size / 1024**2 if p.exists() else np.nan for p in expected_tables + expected_figures],
})
display(audit)


## Load Metrics


In [ ]:
def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if not path.exists():
        display(Markdown(f'**Missing table:** `{rel(path)}`'))
        return pd.DataFrame()
    df = pd.read_csv(path)
    print(f'loaded {rel(path)}: {len(df)} rows, {len(df.columns)} columns')
    return df


def add_generalization_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    df = df.copy()
    for q in ['q50', 'q68', 'q90', 'q95', 'q99']:
        copy_col = f'gen_copy_fraction_{q}'
        gl_col = f'gen_gl_{q}'
        if gl_col not in df.columns and copy_col in df.columns:
            df[gl_col] = 1.0 - df[copy_col]
    # Some older tables used shorter names.
    for q in ['q90', 'q95', 'q99']:
        if f'gen_gl_{q}' not in df.columns and f'copy_fraction_{q}' in df.columns:
            df[f'gen_gl_{q}'] = 1.0 - df[f'copy_fraction_{q}']
    if 'dataset_tag' not in df.columns:
        name_col = 'run_name' if 'run_name' in df.columns else df.columns[0]
        df['dataset_tag'] = df[name_col].map(dataset_tag_from_name)
    if 'dataset_size' not in df.columns:
        df['dataset_size'] = df['dataset_tag'].map(dataset_size_from_tag)
    return df


pca_metrics = add_generalization_columns(read_csv_if_exists(TABLE_DIR / 'nf_generalize_fig2_dit_pca_full_nn_metrics.csv'))
sscd_metrics = add_generalization_columns(read_csv_if_exists(TABLE_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_metrics.csv'))

for feature_name, df in [('PCA', pca_metrics), ('SSCD', sscd_metrics)]:
    display(Markdown(f'### {feature_name} metrics'))
    if df.empty:
        continue
    print('generalization columns:', [c for c in df.columns if c.startswith('gen_gl')])
    preferred = [
        'run_name', 'dataset_tag', 'dataset_size', 'n_generated', 'n_train',
        'gen_gl_q90', 'gen_gl_q95', 'gen_gl_q99',
        'gen_copy_fraction_q95', 'threshold_q95', 'gen_nn_median', 'gen_nn_q95'
    ]
    cols = [c for c in preferred if c in df.columns]
    display(df[cols].sort_values('dataset_size') if cols else df.head())


## Training Loss Curves

This reads the latest checkpoint metrics for each DiT run and plots loss against optimizer update. Loss is not a fidelity metric, but it is useful for catching unfinished or unstable runs before interpreting samples.


In [ ]:

def checkpoint_epoch(path: Path) -> int | None:
    m = re.search(r'checkpoint-epoch-(\d+)', str(path))
    return int(m.group(1)) if m else None


def metric_candidates(row: pd.Series) -> list[Path]:
    root = Path(str(row.get('checkpoint_dir', '') or ''))
    paths: list[Path] = []
    if root.exists():
        paths.extend(sorted(root.glob('metrics_epoch_*.json')))
        metrics_json = root / 'metrics.json'
        if metrics_json.exists():
            paths.append(metrics_json)
        for ckpt in sorted(root.glob('checkpoint-epoch-*')):
            paths.extend(sorted(ckpt.glob('metrics*.json')))
    return paths


def flatten_numeric(values: Any) -> np.ndarray:
    if values is None:
        return np.asarray([], dtype=float)
    out: list[float] = []

    def visit(x: Any) -> None:
        if x is None:
            return
        if isinstance(x, dict):
            for key in ('loss', 'value', 'mean', 'avg'):
                if key in x:
                    visit(x[key])
                    return
            return
        if isinstance(x, (list, tuple, np.ndarray)):
            for item in x:
                visit(item)
            return
        try:
            out.append(float(x))
        except (TypeError, ValueError):
            return

    visit(values)
    return np.asarray(out, dtype=float)


def read_latest_metrics(row: pd.Series) -> tuple[dict[str, Any], Path | None]:
    paths = metric_candidates(row)
    if not paths:
        return {}, None

    def score(path: Path) -> tuple[int, float]:
        epoch = checkpoint_epoch(path)
        return (epoch if epoch is not None else -1, path.stat().st_mtime)

    latest = max(paths, key=score)
    try:
        with latest.open() as f:
            return json.load(f), latest
    except Exception as exc:
        print('failed reading metrics:', latest, exc)
        return {}, latest


def downsample_xy(x: np.ndarray, y: np.ndarray, max_points: int = 1200) -> tuple[np.ndarray, np.ndarray]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if len(x) <= max_points:
        return x, y
    idx = np.linspace(0, len(x) - 1, max_points, dtype=int)
    return x[idx], y[idx]


loss_by_run: dict[str, dict[str, Any]] = {}
loss_rows = []
if manifest_df.empty:
    display(Markdown('No manifest available; skipping loss audit.'))
else:
    for _, row in manifest_df.sort_values('dataset_size').iterrows():
        metrics, metrics_path = read_latest_metrics(row)
        epoch_loss = flatten_numeric(metrics.get('epoch_loss'))
        batch_loss = flatten_numeric(metrics.get('loss', metrics.get('batch_loss')))
        epoch_lr = flatten_numeric(metrics.get('epoch_lr', metrics.get('lr')))
        run_name = str(row.get('run_name'))
        loss_by_run[run_name] = {
            'metrics': metrics,
            'metrics_path': metrics_path,
            'epoch_loss': epoch_loss,
            'batch_loss': batch_loss,
            'epoch_lr': epoch_lr,
        }
        loss_rows.append({
            'run_name': run_name,
            'dataset_tag': row.get('dataset_tag'),
            'dataset_size': int(row.get('dataset_size')) if pd.notna(row.get('dataset_size')) else np.nan,
            'steps_per_epoch': int(row.get('steps_per_epoch', 1) or 1),
            'metrics_path': rel(metrics_path) if metrics_path else None,
            'n_epoch_loss': len(epoch_loss),
            'final_epoch_loss': float(epoch_loss[-1]) if len(epoch_loss) else np.nan,
            'best_epoch_loss': float(np.nanmin(epoch_loss)) if len(epoch_loss) else np.nan,
            'n_batch_loss': len(batch_loss),
            'final_batch_loss': float(batch_loss[-1]) if len(batch_loss) else np.nan,
        })

    loss_df = pd.DataFrame(loss_rows).sort_values('dataset_size')
    display(loss_df)

    if loss_df['n_epoch_loss'].sum() == 0 and loss_df['n_batch_loss'].sum() == 0:
        print('No training metrics JSON found yet.')
    else:
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)
        for _, row in manifest_df.sort_values('dataset_size').iterrows():
            run_name = str(row.get('run_name'))
            info = loss_by_run.get(run_name, {})
            label = rf"$2^{{{int(round(math.log2(float(row['dataset_size']))))}}}$"
            steps_per_epoch = max(1, int(row.get('steps_per_epoch', 1) or 1))

            epoch_loss = np.asarray(info.get('epoch_loss', []), dtype=float)
            if len(epoch_loss):
                x = np.arange(len(epoch_loss), dtype=float) * steps_per_epoch
                x, y = downsample_xy(x, epoch_loss)
                axes[0].plot(x, y, lw=1.5, label=label)

            batch_loss = np.asarray(info.get('batch_loss', []), dtype=float)
            if len(batch_loss):
                x = np.arange(len(batch_loss), dtype=float)
                y = batch_loss
                window = max(1, len(y) // 1200)
                if window > 1:
                    kernel = np.ones(window, dtype=float) / window
                    y = np.convolve(y, kernel, mode='valid')
                    x = x[:len(y)] + 0.5 * (window - 1)
                x, y = downsample_xy(x, y)
                axes[1].plot(x, y, lw=1.2, alpha=0.85, label=label)

            epoch_lr = np.asarray(info.get('epoch_lr', []), dtype=float)
            if len(epoch_lr):
                x = np.arange(len(epoch_lr), dtype=float) * steps_per_epoch
                x, y = downsample_xy(x, epoch_lr)
                axes[2].plot(x, y, lw=1.2, label=label)

        axes[0].set_title('epoch loss')
        axes[0].set_xlabel('optimizer update')
        axes[0].set_ylabel('mean training loss')
        axes[1].set_title('batch loss, smoothed')
        axes[1].set_xlabel('optimizer update')
        axes[1].set_ylabel('training loss')
        axes[2].set_title('learning rate')
        axes[2].set_xlabel('optimizer update')
        axes[2].set_ylabel('LR')
        for ax in axes:
            ax.grid(alpha=0.22)
            if ax.has_data():
                ax.set_yscale('log')
        handles, labels = axes[0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.08), ncol=min(6, len(labels)), frameon=False, title=r'$N_{2D}$')
        out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_training_curves.png'
        QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, bbox_inches='tight')
        plt.show()
        print('wrote', out)


## Load Generated and Real Reference Slices

These cells load the DiT sample `.npz` files and the matching real CAMELS training slices from each run config. The load is capped for notebook responsiveness; increase `DIT_MAX_GENERATED` or `DIT_MAX_REAL_RAW_CUBES` if needed.


In [ ]:

def npz_array_key(path: Path) -> str:
    with np.load(path) as data:
        return 'samples' if 'samples' in data.files else data.files[0]


def load_npz_array(path: Path) -> np.ndarray:
    with np.load(path) as data:
        key = 'samples' if 'samples' in data.files else data.files[0]
        arr = np.asarray(data[key], dtype=np.float32)
    if SIMDIFF_EVAL_AVAILABLE:
        return as_nchw(arr)
    if arr.ndim == 3:
        return arr[:, None, :, :]
    if arr.ndim == 4 and arr.shape[1] in (1, 3):
        return arr
    if arr.ndim == 4 and arr.shape[-1] in (1, 3):
        return np.moveaxis(arr, -1, 1)
    raise ValueError(f'Could not interpret generated array shape {arr.shape}')


def evenly_limit(arr: np.ndarray, limit: int | None) -> np.ndarray:
    arr = np.asarray(arr)
    if limit is None or len(arr) <= int(limit):
        return arr.copy()
    idx = np.linspace(0, len(arr) - 1, int(limit), dtype=int)
    return arr[idx].copy()


def config_path_for(row: pd.Series) -> Path:
    raw = str(row.get('config', '') or row.get('config_path', '') or '')
    if raw:
        path = Path(raw)
        return path if path.is_absolute() else PROJECT_DIR / path
    return PROJECT_DIR / 'local' / SWEEP_NAME / 'configs' / f"{row['run_name']}.yaml"


loaded: dict[str, dict[str, Any]] = {}
load_rows = []
if manifest_df.empty:
    display(Markdown('No manifest available; skipping generated/real loading.'))
elif not SIMDIFF_EVAL_AVAILABLE:
    display(Markdown(f'`simdiff_eval` unavailable, so real-reference diagnostics are skipped: `{SIMDIFF_EVAL_ERROR}`'))
else:
    for _, row in manifest_df.sort_values('dataset_size').iterrows():
        sample_path = Path(row['sample_path_resolved'])
        if not sample_path.exists():
            continue
        cfg_path = config_path_for(row)
        try:
            generated = evenly_limit(load_npz_array(sample_path), MAX_GENERATED)
            raw_cap = min(int(row.get('n_train_simulations', MAX_REAL_RAW_CUBES) or MAX_REAL_RAW_CUBES), MAX_REAL_RAW_CUBES)
            real = as_nchw(load_real_from_config(cfg_path, max_raw_samples=raw_cap))
            run_name = str(row.get('run_name'))
            loaded[run_name] = {
                'spec': row,
                'real': real,
                'generated': generated,
                'sample_path': sample_path,
                'config_path': cfg_path,
            }
            load_rows.append({
                'run_name': run_name,
                'dataset_tag': row.get('dataset_tag'),
                'dataset_size': int(row.get('dataset_size')),
                'n_real_loaded': len(real),
                'n_generated_loaded': len(generated),
                'sample_path': rel(sample_path),
                'config_path': rel(cfg_path),
            })
        except Exception as exc:
            load_rows.append({
                'run_name': row.get('run_name'),
                'dataset_tag': row.get('dataset_tag'),
                'dataset_size': row.get('dataset_size'),
                'n_real_loaded': 0,
                'n_generated_loaded': 0,
                'error': repr(exc),
                'sample_path': rel(sample_path),
                'config_path': rel(cfg_path),
            })

loaded_df = pd.DataFrame(load_rows).sort_values('dataset_size') if load_rows else pd.DataFrame()
display(loaded_df)
print('loaded DiT sample rows:', len(loaded))


## Generated Image Grids Across Data Size

A quick visual check across the DiT data-size sweep. These are not nearest-neighbor diagnostics; they just show what the generated fields look like as `N_2D` changes.


In [ ]:

def dataset_size_label(n: int) -> str:
    n = int(n)
    log2n = np.log2(n)
    if np.isfinite(log2n) and abs(log2n - round(log2n)) < 1e-9:
        return rf'$2^{{{int(round(log2n))}}}$'
    return f'{n:,}'


def choose_bundles(tags: list[str], max_count: int | None = None) -> list[dict[str, Any]]:
    bundles = [b for b in loaded.values() if str(b['spec'].get('dataset_tag')) in set(tags)]
    bundles = sorted(bundles, key=lambda b: int(b['spec']['dataset_size']))
    if not bundles:
        bundles = sorted(loaded.values(), key=lambda b: int(b['spec']['dataset_size']))
    if max_count is not None and len(bundles) > max_count:
        keep = np.linspace(0, len(bundles) - 1, max_count, dtype=int)
        bundles = [bundles[i] for i in keep]
    return bundles


def plot_dit_image_grid(sample_index: int = 0, tags: list[str] = IMAGE_TAGS) -> Path | None:
    if not loaded:
        display(Markdown('No loaded DiT samples available for image grid.'))
        return None
    bundles = choose_bundles(tags, max_count=6)
    if not bundles:
        display(Markdown('No selected bundles available for image grid.'))
        return None

    values = []
    for b in bundles:
        values.append(b['generated'][min(sample_index, len(b['generated']) - 1), 0].ravel())
        values.append(b['real'][0, 0].ravel())
    flat = np.concatenate(values)
    vmin = float(np.nanquantile(flat, 0.005))
    vmax = float(np.nanquantile(flat, 0.995))

    fig, axes = plt.subplots(2, len(bundles), figsize=(2.7 * len(bundles), 5.8), squeeze=False, constrained_layout=True)
    for col, b in enumerate(bundles):
        row = b['spec']
        gen_idx = min(sample_index, len(b['generated']) - 1)
        axes[0, col].imshow(b['generated'][gen_idx, 0], cmap='viridis', vmin=vmin, vmax=vmax)
        axes[1, col].imshow(b['real'][0, 0], cmap='viridis', vmin=vmin, vmax=vmax)
        axes[0, col].set_title(dataset_size_label(int(row['dataset_size'])))
        for ax in axes[:, col]:
            ax.set_xticks([])
            ax.set_yticks([])
    axes[0, 0].set_ylabel('generated', fontsize=15, fontweight='bold')
    axes[1, 0].set_ylabel('real reference', fontsize=15, fontweight='bold')
    fig.suptitle('DiT-base generated maps across training-set size', y=1.03)
    out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_generated_image_grid.png'
    fig.savefig(out, bbox_inches='tight')
    plt.show()
    print('wrote', out)
    return out

image_grid_path = plot_dit_image_grid(sample_index=int(os.environ.get('DIT_IMAGE_SAMPLE_INDEX', '0')))


## One-point and P(k) Fidelity Across Data Size

These panels ask a different question from memorization: do the generated samples match real CAMELS summary statistics? Small-data runs can look good here by copying training slices, so read this together with the PCA/SSCD generalization curves.


In [ ]:

def plot_dit_onepoint_pk(tags: list[str] = DETAIL_TAGS) -> Path | None:
    if not loaded:
        display(Markdown('No loaded DiT samples available for one-point/P(k) plots.'))
        return None
    if not SIMDIFF_EVAL_AVAILABLE:
        display(Markdown('`simdiff_eval` unavailable; cannot compute one-point/P(k) diagnostics.'))
        return None

    bundles = choose_bundles(tags, max_count=5)
    if not bundles:
        display(Markdown('No selected DiT bundles available for one-point/P(k) plots.'))
        return None

    n = len(bundles)
    fig, axes = plt.subplots(2, n, figsize=(max(4.2 * n, 8.6), 7.4), squeeze=False, constrained_layout=True)
    rows = []
    for col, bundle in enumerate(bundles):
        row = bundle['spec']
        real = bundle['real']
        generated = bundle['generated']

        rh = field_histogram(real, bins=140)
        gh = field_histogram(generated, bins=140)
        edges = np.asarray(rh['bin_edges'])
        centers = 0.5 * (edges[:-1] + edges[1:])
        axes[0, col].plot(centers, rh['hist'], color='black', lw=2.2, label='real')
        axes[0, col].plot(centers, gh['hist'], color='#0072B2', lw=2.0, label='DiT generated')
        axes[0, col].set_yscale('log')
        axes[0, col].set_title(f"{dataset_size_label(int(row['dataset_size']))} one-point")
        axes[0, col].set_xlabel('Normalized field value')
        if col == 0:
            axes[0, col].set_ylabel('Pixel PDF')
            axes[0, col].legend(frameon=False, loc='upper right')

        pk_real, kbins = batch_power_spectra(real, nbins=PK_NBINS)
        pk_gen, _ = batch_power_spectra(generated, nbins=PK_NBINS)
        mean_real = np.clip(np.nanmean(pk_real, axis=0), 1e-30, None)
        mean_gen = np.nanmean(pk_gen, axis=0)
        ratio = mean_gen / mean_real
        axes[1, col].plot(kbins, ratio, marker='o', ms=4.4, lw=2.0, color='#0072B2')
        axes[1, col].axhline(1.0, color='0.25', ls='--', lw=1.4)
        upper = max(2.0, float(np.nanquantile(ratio, 0.98)) * 1.15 if np.isfinite(ratio).any() else 2.0)
        axes[1, col].set_ylim(0, upper)
        axes[1, col].set_title('Mean $P(k)$ ratio')
        axes[1, col].set_xlabel('$k$ bin')
        if col == 0:
            axes[1, col].set_ylabel('generated / real')

        finite = np.isfinite(ratio)
        rows.append({
            'run_name': row.get('run_name'),
            'dataset_tag': row.get('dataset_tag'),
            'dataset_size': int(row['dataset_size']),
            'pk_ratio_median': float(np.nanmedian(ratio[finite])) if finite.any() else np.nan,
            'pk_ratio_min': float(np.nanmin(ratio[finite])) if finite.any() else np.nan,
            'pk_ratio_max': float(np.nanmax(ratio[finite])) if finite.any() else np.nan,
            'max_abs_pk_ratio_minus_1': float(np.nanmax(np.abs(ratio[finite] - 1.0))) if finite.any() else np.nan,
        })

        for ax in axes[:, col]:
            ax.grid(alpha=0.18)
            for spine in ['top', 'right']:
                ax.spines[spine].set_visible(False)

    fig.suptitle('DiT-base physical-statistics check', y=1.03)
    out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_detail_onepoint_pk.png'
    fig.savefig(out, bbox_inches='tight')
    plt.show()
    print('wrote', out)

    fidelity_summary = pd.DataFrame(rows).sort_values('dataset_size')
    display(fidelity_summary)
    table_out = TABLE_DIR / 'nf_generalize_fig2_dit_fidelity_summary.csv'
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    fidelity_summary.to_csv(table_out, index=False)
    print('wrote', table_out)
    return out

fidelity_plot_path = plot_dit_onepoint_pk()


## DiT Generalization Curves

The main DiT readout is the generalization score versus training-set size. A score near zero means generated fields are often too close to training slices under the chosen embedding. A score near one means generated fields are not unusually close to the training set relative to the real-data baseline.


In [ ]:
def format_power_ticks(ax, values):
    vals = sorted({int(v) for v in values if pd.notna(v) and v > 0})
    if not vals:
        return
    ax.set_xscale('log', base=2)
    ax.set_xticks(vals)
    ax.set_xticklabels([rf'$2^{{{int(round(math.log2(v)))}}}$' for v in vals])


def plot_dit_generalization_curves(metrics_by_feature: dict[str, pd.DataFrame], quantile: str = 'q95') -> Path | None:
    gl_col = f'gen_gl_{quantile}'
    fig, ax = plt.subplots(figsize=(8.6, 5.4), constrained_layout=True)
    colors = {'PCA': '#0072B2', 'SSCD': '#D55E00'}
    markers = {'PCA': 'o', 'SSCD': 's'}
    all_x = []
    plotted = False

    for feature_name, df in metrics_by_feature.items():
        if df.empty or gl_col not in df.columns or 'dataset_size' not in df.columns:
            continue
        sub = df.dropna(subset=['dataset_size', gl_col]).sort_values('dataset_size')
        if sub.empty:
            continue
        all_x.extend(sub['dataset_size'].astype(float).tolist())
        ax.plot(
            sub['dataset_size'], sub[gl_col],
            marker=markers.get(feature_name, 'o'), ms=8, lw=3,
            color=colors.get(feature_name), label=feature_name,
        )
        plotted = True

    if not plotted:
        display(Markdown(f'No `{gl_col}` columns found to plot.'))
        plt.close(fig)
        return None

    format_power_ticks(ax, all_x)
    ax.axhline(0.5, color='0.35', lw=1.5, ls=':', label='0.5 transition marker')
    ax.set_ylim(-0.04, 1.04)
    ax.set_xlabel(r'Training set size $N_{2D}$')
    ax.set_ylabel('Generalization score')
    ax.set_title(f'DiT-base memorization-to-generalization check ({quantile})')
    ax.grid(True, alpha=0.22)
    ax.legend(frameon=False, loc='lower right')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

    QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)
    out = QUICKCHECK_DIR / f'nf_generalize_fig2_dit_combined_gl_curves_{quantile}.png'
    fig.savefig(out, bbox_inches='tight')
    plt.show()
    print('wrote', out)
    return out

combined_curve = plot_dit_generalization_curves({'PCA': pca_metrics, 'SSCD': sscd_metrics}, quantile='q95')


## Transition Summary

`N50` is the interpolated training-set size where the generalization score crosses 0.5. This is a compact way to compare transitions, but it should be read with the full curve because a single midpoint can hide changes in slope or tail behavior.


In [ ]:
def interpolate_crossing(df: pd.DataFrame, ycol: str, threshold: float = 0.5) -> dict[str, Any]:
    if df.empty or ycol not in df.columns or 'dataset_size' not in df.columns:
        return {'status': 'missing', 'n_cross': np.nan, 'log2_n_cross': np.nan}
    sub = df[['dataset_size', ycol]].dropna().sort_values('dataset_size')
    sub = sub[sub['dataset_size'] > 0]
    if sub.empty:
        return {'status': 'missing', 'n_cross': np.nan, 'log2_n_cross': np.nan}

    x = np.log2(sub['dataset_size'].astype(float).to_numpy())
    y = sub[ycol].astype(float).to_numpy()
    if y[0] >= threshold:
        return {'status': 'left_censored', 'n_cross': 2 ** x[0], 'log2_n_cross': x[0]}
    if y[-1] < threshold:
        return {'status': 'right_censored', 'n_cross': 2 ** x[-1], 'log2_n_cross': x[-1]}

    for i in range(len(y) - 1):
        y0, y1 = y[i], y[i + 1]
        if (y0 <= threshold <= y1) or (y1 <= threshold <= y0):
            if y1 == y0:
                xc = x[i]
            else:
                frac = (threshold - y0) / (y1 - y0)
                xc = x[i] + frac * (x[i + 1] - x[i])
            return {'status': 'interpolated', 'n_cross': 2 ** xc, 'log2_n_cross': xc}
    return {'status': 'not_found', 'n_cross': np.nan, 'log2_n_cross': np.nan}


rows = []
for feature_name, df in [('PCA', pca_metrics), ('SSCD', sscd_metrics)]:
    for q in ['q90', 'q95', 'q99']:
        col = f'gen_gl_{q}'
        result = interpolate_crossing(df, col, threshold=0.5)
        rows.append({
            'feature': feature_name,
            'score_col': col,
            'threshold': 0.5,
            **result,
        })

transition_df = pd.DataFrame(rows)
display(transition_df)

if len(transition_df):
    out = TABLE_DIR / 'nf_generalize_fig2_dit_transition_summary.csv'
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    transition_df.to_csv(out, index=False)
    print('wrote', out)


## Compare DiT With Existing UNet Sweep (Optional)

This section runs only if the original UNet Fig.2 PCA/SSCD tables are present in `results/nf_generalize_fig2/tables/`. It is useful for checking whether the Transformer transition is left-shifted, right-shifted, or similar to the UNet baselines.


In [ ]:
UNET_RESULTS_DIR = PROJECT_DIR / 'results' / 'nf_generalize_fig2'
UNET_TABLE_DIR = UNET_RESULTS_DIR / 'tables'

unet_pca = add_generalization_columns(read_csv_if_exists(UNET_TABLE_DIR / 'nf_generalize_fig2_pca_full_nn_metrics.csv'))
unet_sscd = add_generalization_columns(read_csv_if_exists(UNET_TABLE_DIR / 'nf_generalize_fig2_sscd_full_nn_metrics.csv'))


def arch_label(raw: Any) -> str:
    text = str(raw)
    mapping = {'u64': 'UNet-64', 'u128': 'UNet-128', 'u256': 'UNet-256', 'dit': 'DiT-base', 'dit_base': 'DiT-base'}
    return mapping.get(text, text)


def infer_arch_column(df: pd.DataFrame) -> pd.Series:
    if 'arch' in df.columns:
        return df['arch'].astype(str)
    if 'arch_label' in df.columns:
        return df['arch_label'].astype(str)
    if 'run_name' in df.columns:
        def from_name(name):
            m = re.search(r'(u64|u128|u256|dit_base|dit)', str(name))
            return m.group(1) if m else 'unknown'
        return df['run_name'].map(from_name)
    return pd.Series(['unknown'] * len(df), index=df.index)


def plot_dit_vs_unet(feature_name: str, dit_df: pd.DataFrame, unet_df: pd.DataFrame, quantile: str = 'q95') -> Path | None:
    gl_col = f'gen_gl_{quantile}'
    if dit_df.empty or gl_col not in dit_df.columns:
        display(Markdown(f'No DiT `{feature_name}` `{gl_col}` data.'))
        return None

    fig, ax = plt.subplots(figsize=(9.0, 5.6), constrained_layout=True)
    all_x = []
    colors = {'u64': '#009E73', 'u128': '#D55E00', 'u256': '#0072B2'}
    markers = {'u64': '^', 'u128': 'o', 'u256': 's'}

    if not unet_df.empty and gl_col in unet_df.columns:
        tmp = unet_df.copy()
        tmp['arch_for_plot'] = infer_arch_column(tmp)
        for arch in ['u64', 'u128', 'u256']:
            sub = tmp[tmp['arch_for_plot'].astype(str) == arch].dropna(subset=['dataset_size', gl_col]).sort_values('dataset_size')
            if sub.empty:
                continue
            all_x.extend(sub['dataset_size'].astype(float).tolist())
            ax.plot(
                sub['dataset_size'], sub[gl_col],
                color=colors[arch], marker=markers[arch], lw=2.2, ms=7,
                alpha=0.55, label=arch_label(arch),
            )

    sub = dit_df.dropna(subset=['dataset_size', gl_col]).sort_values('dataset_size')
    all_x.extend(sub['dataset_size'].astype(float).tolist())
    ax.plot(
        sub['dataset_size'], sub[gl_col],
        color='black', marker='D', lw=3.4, ms=8,
        label='DiT-base',
    )
    ax.axhline(0.5, color='0.35', lw=1.4, ls=':')
    format_power_ticks(ax, all_x)
    ax.set_ylim(-0.04, 1.04)
    ax.set_xlabel(r'Training set size $N_{2D}$')
    ax.set_ylabel('Generalization score')
    ax.set_title(f'{feature_name}: DiT-base compared with UNet baselines ({quantile})')
    ax.grid(True, alpha=0.22)
    ax.legend(frameon=False, ncol=2, loc='lower right')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

    out = QUICKCHECK_DIR / f'nf_generalize_fig2_dit_vs_unet_{feature_name.lower()}_{quantile}.png'
    fig.savefig(out, bbox_inches='tight')
    plt.show()
    print('wrote', out)
    return out

_ = plot_dit_vs_unet('PCA', pca_metrics, unet_pca, quantile='q95')
_ = plot_dit_vs_unet('SSCD', sscd_metrics, unet_sscd, quantile='q95')


## Capacity Check: Is DiT Just Bigger?

DiT-base is deliberately close to the UNet-128 parameter count. That makes the comparison sharper: if DiT-base transitions later than UNet-128, the difference is not simply because DiT has many more trainable weights.

This section estimates the q95 `N50` transition point for each architecture, then compares it to model parameter count. Treat this as a diagnostic, not a fitted scaling law, because we only have one DiT capacity.


In [ ]:
# Parameter counts from the tracked config templates / print_model_param_count utility.
# DiT-base is ViT-like, but this is the diffusion-transformer model used here.
MODEL_CAPACITY = pd.DataFrame([
    {'model': 'UNet-64',  'family': 'UNet', 'arch_key': 'u64',      'model_params': 26_621_057},
    {'model': 'UNet-128', 'family': 'UNet', 'arch_key': 'u128',     'model_params': 140_539_521},
    {'model': 'UNet-256', 'family': 'UNet', 'arch_key': 'u256',     'model_params': 196_059_905},
    {'model': 'DiT-base', 'family': 'DiT',  'arch_key': 'dit_base', 'model_params': 138_290_000},
])
MODEL_CAPACITY['model_params_m'] = MODEL_CAPACITY['model_params'] / 1e6

def n50_for_model(feature_name: str, model: str, family: str, model_params: int, df: pd.DataFrame, q: str = 'q95') -> dict[str, Any]:
    col = f'gen_gl_{q}'
    cross = interpolate_crossing(df, col, threshold=0.5)
    return {
        'feature': feature_name,
        'model': model,
        'family': family,
        'model_params': model_params,
        'model_params_m': model_params / 1e6,
        'score_col': col,
        **cross,
    }


def capacity_transition_table(q: str = 'q95') -> pd.DataFrame:
    rows = []
    feature_frames = {
        'PCA': (pca_metrics, unet_pca),
        'SSCD': (sscd_metrics, unet_sscd),
    }
    for feature_name, (dit_df, unet_df) in feature_frames.items():
        dit_params = int(MODEL_CAPACITY.loc[MODEL_CAPACITY['model'] == 'DiT-base', 'model_params'].iloc[0])
        rows.append(n50_for_model(feature_name, 'DiT-base', 'DiT', dit_params, dit_df, q=q))

        if unet_df.empty:
            continue
        tmp = unet_df.copy()
        tmp['arch_for_plot'] = infer_arch_column(tmp)
        for arch in ['u64', 'u128', 'u256']:
            cap = MODEL_CAPACITY[MODEL_CAPACITY['arch_key'] == arch].iloc[0]
            sub = tmp[tmp['arch_for_plot'].astype(str) == arch]
            rows.append(n50_for_model(feature_name, cap['model'], cap['family'], int(cap['model_params']), sub, q=q))
    out = pd.DataFrame(rows)
    return out.sort_values(['feature', 'family', 'model_params']).reset_index(drop=True)


capacity_n50 = capacity_transition_table(q='q95')
display(capacity_n50[['feature', 'model', 'family', 'model_params_m', 'status', 'n_cross', 'log2_n_cross']])

# Pairwise comparison against the parameter-matched UNet-128 and the larger UNet-256.
ratio_rows = []
for feature_name in ['PCA', 'SSCD']:
    sub = capacity_n50[capacity_n50['feature'] == feature_name].set_index('model')
    if 'DiT-base' not in sub.index:
        continue
    for baseline in ['UNet-128', 'UNet-256']:
        if baseline not in sub.index:
            continue
        ratio_rows.append({
            'feature': feature_name,
            'comparison': f'DiT-base / {baseline}',
            'param_ratio': sub.loc['DiT-base', 'model_params'] / sub.loc[baseline, 'model_params'],
            'n50_ratio': sub.loc['DiT-base', 'n_cross'] / sub.loc[baseline, 'n_cross'],
            'dit_log2_n50': sub.loc['DiT-base', 'log2_n_cross'],
            'baseline_log2_n50': sub.loc[baseline, 'log2_n_cross'],
        })
capacity_ratios = pd.DataFrame(ratio_rows)
display(capacity_ratios)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2), sharey=True, constrained_layout=True)
feature_colors = {'PCA': '#0072B2', 'SSCD': '#D55E00'}
for ax, feature_name in zip(axes, ['PCA', 'SSCD']):
    sub = capacity_n50[(capacity_n50['feature'] == feature_name) & capacity_n50['n_cross'].notna()].copy()
    if sub.empty:
        ax.set_visible(False)
        continue
    unet = sub[sub['family'] == 'UNet'].sort_values('model_params')
    dit = sub[sub['family'] == 'DiT']
    if len(unet):
        ax.plot(unet['model_params'], unet['n_cross'], color=feature_colors[feature_name], marker='o', lw=2.4, ms=8, label='UNet family')
        for _, row in unet.iterrows():
            ax.annotate(row['model'].replace('UNet-', 'U'), (row['model_params'], row['n_cross']), xytext=(5, 4), textcoords='offset points', fontsize=10)
    if len(dit):
        row = dit.iloc[0]
        ax.scatter(row['model_params'], row['n_cross'], color='black', marker='D', s=90, label='DiT-base')
        ax.annotate('DiT-base', (row['model_params'], row['n_cross']), xytext=(6, -12), textcoords='offset points', fontsize=10, color='black')
    ax.set_xscale('log')
    ax.set_yscale('log', base=2)
    ax.set_xlabel('Trainable parameters')
    ax.set_title(f'{feature_name}: q95 transition vs capacity')
    ax.grid(True, alpha=0.22)
    ax.legend(frameon=False, loc='upper left')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
axes[0].set_ylabel(r'$N_{50}$ training images')

out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_capacity_n50_q95.png'
fig.savefig(out, bbox_inches='tight')
plt.show()
print('wrote', out)

# Reader-facing interpretation, generated from the table above.
if not capacity_ratios.empty:
    lines = ['### Capacity interpretation']
    for _, row in capacity_ratios.iterrows():
        lines.append(
            f"- {row['feature']}: {row['comparison']} has {row['param_ratio']:.2f}x parameters "
            f"but {row['n50_ratio']:.2f}x the q95 N50."
        )
    lines.append('- If DiT-base is close to UNet-128 in parameter count but needs more images to leave the copy-like regime, that points to architecture/inductive-bias differences rather than raw parameter count alone.')
    display(Markdown('\n'.join(lines)))


## Existing Quickcheck Figures and Saved Diagnostics


In [ ]:
def show_existing_figure(path: Path, title: str, width: int = 950) -> None:
    display(Markdown(f'### {title}'))
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        display(Markdown(f'Missing: `{rel(path)}`'))

show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_training_curves.png', 'DiT training curves')
show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_generated_image_grid.png', 'DiT generated image grid')
show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_detail_onepoint_pk.png', 'DiT one-point and P(k) fidelity')
show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_paper_style_gl_curves.png', 'PCA paper-style GL curves')
show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_paper_style_gl_curves.png', 'SSCD paper-style GL curves')
show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_similarity_curves.png', 'PCA similarity curves')
show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_similarity_curves.png', 'SSCD similarity curves')
show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_copy_fraction_curves.png', 'PCA copy-fraction curves')
show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_copy_fraction_curves.png', 'SSCD copy-fraction curves')


## Sample File Sanity Check

This inspects array keys and shapes without doing the expensive nearest-neighbor analysis again.


In [ ]:
def inspect_npz(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {'exists': False}
    with np.load(path) as data:
        keys = list(data.files)
        first = keys[0] if keys else None
        arr = data[first] if first else None
        return {
            'exists': True,
            'keys': ', '.join(keys[:8]),
            'first_key': first,
            'shape': tuple(arr.shape) if arr is not None else None,
            'dtype': str(arr.dtype) if arr is not None else None,
            'size_mb': path.stat().st_size / 1024**2,
        }

if manifest_df.empty:
    display(Markdown('No manifest available, so sample files were not inspected.'))
else:
    rows = []
    for _, row in manifest_df.sort_values('dataset_size').iterrows():
        path = row['sample_path_resolved']
        info = inspect_npz(path)
        rows.append({
            'dataset_tag': row.get('dataset_tag'),
            'dataset_size': row.get('dataset_size'),
            'path': rel(path),
            **info,
        })
    sample_inspect_df = pd.DataFrame(rows)
    display(sample_inspect_df)


## Takeaways


In [ ]:
def best_transition_line(feature: str) -> str:
    if transition_df.empty:
        return f'- {feature}: transition table missing.'
    sub = transition_df[(transition_df['feature'] == feature) & (transition_df['score_col'] == 'gen_gl_q95')]
    if sub.empty:
        return f'- {feature}: q95 generalization column missing.'
    row = sub.iloc[0]
    if pd.isna(row['n_cross']):
        return f'- {feature}: N50 not available ({row["status"]}).'
    return f'- {feature}: q95 N50 = 2^{row["log2_n_cross"]:.2f} = {row["n_cross"]:.0f} 2D images ({row["status"]}).'

sample_ok = None if manifest_df.empty else int(manifest_df['sample_exists'].sum())
sample_total = None if manifest_df.empty else len(manifest_df)

lines = [
    '### Notebook summary',
]
if sample_ok is not None:
    lines.append(f'- Sample audit: {sample_ok}/{sample_total} DiT sample files found.')
lines.append(best_transition_line('PCA'))
lines.append(best_transition_line('SSCD'))
lines.extend([
    '- Read PCA and SSCD together. PCA is sensitive to low-dimensional variance; SSCD is a learned image-similarity embedding. Agreement is stronger evidence than either diagnostic alone.',
    '- Next check: compare these DiT curves against the UNet curves above. If DiT shifts the transition, then architecture matters beyond parameter count.',
])

display(Markdown('\n'.join(lines)))


## Great Lakes Rerun Command

From the repo root on Great Lakes:

```bash
cd /home/jiamingp/diffusion_models_repo
jupyter nbconvert --execute --to notebook --inplace notebooks/nf_generalize_fig2_dit_results.ipynb
```

If you are using the Jupyter web session, just open this notebook and run all cells. The heavy PCA/SSCD nearest-neighbor work should not rerun here; this notebook reads the completed CSV and PNG outputs.
